# Lakeside heating DSM — PyTorch ResMLP (hybrid component A alt)

Trains / reports **ResMLP multi-head** on the **real BAS 15-min store only**.
Does **not** overwrite the hybrid desktop walk or sklearn ExtraTrees ship.

| | |
|---|---|
| Data | `REAL_BAS_15MIN` site store |
| Artifact | `real_baseline_15min_torch_v1.*` |
| Honesty | `HYBRID_SCREENING` |

No proxy labels. No hourly torch v1 ship.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "ml"))

from artifact_paths import artifact_paths
from notebook_proof import prove_real_store_load
from notebook_plots import save_fig

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
print("ROOT", ROOT)
print("SITE", SITE)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE)

## 1 · Proof — real store only

In [ ]:
real_df, meta = prove_real_store_load(site=SITE)
assert (real_df["provenance"] == "REAL_BAS_15MIN").all()
winter = real_df[real_df["month"].isin([11, 12, 1, 2, 3])]
print("winter rows", len(winter), "days", winter["day"].nunique())

## 2 · Load ResMLP card (trained via `train_real_baseline_torch_15min.py`)

In [ ]:
card_path = PATHS["figures"].parent / "real_baseline_15min_torch_v1_model_card.json"
assert card_path.is_file(), f"missing {card_path} — run ml/train_real_baseline_torch_15min.py"
card = json.loads(card_path.read_text(encoding="utf-8"))
assert card.get("honesty") == "HYBRID_SCREENING"
assert card.get("family") == "resmlp_multihead"

sk_card = json.loads(PATHS["real_baseline_card"].read_text(encoding="utf-8"))
sk_peak = sk_card["cv_teacher_forced"][sk_card["champion"]].get(
    "facility_kw_mae_peak_05_09",
    sk_card["cv_teacher_forced"][sk_card["champion"]].get("facility_kw", {}).get("mae_peak_05_09"),
)
torch_peak = card["cv_teacher_forced"]["facility_kw_mae_peak_05_09"]

cmp = pd.DataFrame([
    {"family": f"sklearn_{sk_card['champion']}", "mae_peak_05_09": sk_peak},
    {"family": "resmlp_multihead", "mae_peak_05_09": torch_peak},
]).sort_values("mae_peak_05_09")
display(cmp.round(3))
display(Markdown(
    f"ResMLP peak MAE **{torch_peak:.1f} kW** vs sklearn ship **{sk_peak:.1f} kW**. "
    f"Desktop hybrid walk stays on sklearn ExtraTrees + RF delta."
))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(cmp["family"], cmp["mae_peak_05_09"], color=["#2a9d8f", "#e76f51"])
ax.set_xlabel("Peak MAE kW (HE 05–09)")
ax.set_title("Real baseline — sklearn ship vs ResMLP alt")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(PATHS["figures"] / "hybrid_torch_vs_sklearn.png", fig)
plt.show()
print("card", card_path)

## Honesty

- Trained on measured BAS only (`REAL_BAS_15MIN`).
- Does not write `heating_dsm_hourly_torch_v1` (quarantined).
- Hybrid desktop ship = `promote_hybrid_ship.py` walk JSON.